In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split 
from sklearn.metrics import roc_auc_score

from lightgbm import LGBMClassifier

# Read data files
x = pd.read_csv("../data/train_encoded.csv")
y = pd.read_csv("../data/target.csv").squeeze()
x_test = pd.read_csv("../data/test_encoded.csv")

# Test train split from sklearn model selection
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# LGBM Model configuration
model = LGBMClassifier(
    objective="binary",                 # Defines that it is a 2 state problem
    n_estimators=1000,                  # Number of trees that can be generated
    learning_rate=0.05,                 # Learning Rate
    num_leaves=31,                      # Size of each tree
    random_state=42                     # Ensures reproducability
)

# Training model
model.fit(
    x_train,                        # Training Features
    y_train,                        # Solutions to those features
    eval_set=[(x_val, y_val)],      # Evaluate performance for validation data
    eval_metric="auc"               # Specify ROC AUC as the performance metric
)

# Generating validation prediction
# Makes predictions and classifies them as probabilities over flat integers
# Also uses [:,1] as it returns two probabilitiy rows for no pit vs pit
val_prediction = model.predict_proba(x_val)[:,1]       

# Calculates auc score to evaluate performance of the model and prints
auc = roc_auc_score(y_val, val_prediction)
print(f"AUC: {auc:.4f}")

# Generate predictions from test dataframe for submission
test_prediction = model.predict_proba(x_test)[:,1]

# Create submission 
raw_test = pd.read_csv("../data/test.csv")

submission = pd.DataFrame({

    "id": raw_test["id"],
    "PitNextLap": test_prediction
})

submission.to_csv("../submissions/lgbm_baseline_sub.csv", index=False)

AUC: 0.9482


In [ ]:
check = pd.read_csv("../submissions/lgbm_baseline_sub.csv")

print(check.shape)
check.head()

(188165, 2)


,id,PitNextLap
0,439140,0.004084
1,439141,0.001952
2,439142,0.003651
3,439143,0.143164
4,439144,0.822525
